<a href="https://colab.research.google.com/github/rdelhibabu/FL-BR_H5.0/blob/main/FL_BR_H5_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# RECONFIGMED-CHAIN: Core Federated Optimization & DP Simulation
# Author: Radhakrishnan Delhibabu (rdelhibabu@vit.ac.in)
# Architecture: Healthcare 5.0 Trustless Digital Twin Synchronization
# ==============================================================================

!pip install torch torchvision opacus pandas numpy scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from opacus import PrivacyEngine
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# --- Hyperparameters (from Table 4) ---
NUM_CLIENTS = 10
ROUNDS = 80
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
LR = 1e-3
MAX_GRAD_NORM = 1.0 # C
NOISE_MULTIPLIER = 0.7 # sigma
TARGET_EPSILON = 5.0
ATTENTION_TEMP = 4.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. L-CNN Model Definition (L2 Edge Feature Extraction) ---
class EdgeLCNN(nn.Module):
    def __init__(self, input_dim=32):
        super(EdgeLCNN, self).__init__()
        # Depthwise separable convolutions for edge efficiency
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1, groups=1)
        self.depthwise = nn.Conv1d(16, 16, kernel_size=3, padding=1, groups=16)
        self.pointwise = nn.Conv1d(16, 32, kernel_size=1)
        self.fc1 = nn.Linear(32 * input_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.unsqueeze(1) # Add channel dimension
        x = F.relu(self.conv1(x))
        x = F.relu(self.pointwise(self.depthwise(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))

# --- 2. Attention Mechanism (L3 Federated Aggregation) ---
class AttentionAggregator(nn.Module):
    def __init__(self, state_dim=4):
        super(AttentionAggregator, self).__init__()
        # 2-layer MLP for attention weights
        self.mlp = nn.Sequential(
            nn.Linear(state_dim, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )

    def forward(self, client_states, temperature):
        # client_states shape: [num_clients, state_dim]
        scores = self.mlp(client_states) * temperature
        weights = F.softmax(scores, dim=0)
        return weights

# --- 3. Mock Data Generation (MIMIC-IV Sepsis Proxy) ---
def generate_non_iid_data(num_clients, samples_per_client=1000, input_dim=32):
    client_loaders = []
    # Simulate Dirichlet distribution skew for sepsis positive class
    prevalence_rates = np.random.dirichlet(np.ones(num_clients) * 0.5) * 10

    for k in range(num_clients):
        rate = max(min(prevalence_rates[k], 0.34), 0.07) # Bound between 7% and 34%
        X = torch.randn(samples_per_client, input_dim)
        y = (torch.rand(samples_per_client, 1) < rate).float()
        dataset = TensorDataset(X, y)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
        client_loaders.append(loader)
    return client_loaders

# --- 4. Simulated Federated Loop (Algorithm 1) ---
def federated_training_loop():
    print("Initializing RECONFIGMED-CHAIN Federation...")
    global_model = EdgeLCNN().to(device)
    client_loaders = generate_non_iid_data(NUM_CLIENTS)
    attention_module = AttentionAggregator().to(device)

    # State tracking: [log(n_k), mean_loss, delta_loss, drift_indicator]
    client_states = torch.zeros((NUM_CLIENTS, 4)).to(device)

    for round_num in range(ROUNDS):
        local_weights = []
        local_losses = []

        for k in range(NUM_CLIENTS):
            # 1. Hot-swap local model with global weights
            local_model = EdgeLCNN().to(device)
            local_model.load_state_dict(global_model.state_dict())
            optimizer = optim.Adam(local_model.parameters(), lr=LR)
            criterion = nn.BCELoss()

            # 2. Attach Differential Privacy Engine (L2)
            privacy_engine = PrivacyEngine()
            local_model, optimizer, train_loader = privacy_engine.make_private(
                module=local_model,
                optimizer=optimizer,
                data_loader=client_loaders[k],
                noise_multiplier=NOISE_MULTIPLIER,
                max_grad_norm=MAX_GRAD_NORM,
            )

            # 3. Local Training Execution
            local_model.train()
            epoch_loss = 0
            for epoch in range(LOCAL_EPOCHS):
                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    output = local_model(X_batch)
                    loss = criterion(output, y_batch)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()

            avg_loss = epoch_loss / (LOCAL_EPOCHS * len(train_loader))

            # --- THE FIX ---
            # Opacus prepends '_module.' to all state_dict keys.
            # We remove it here so it matches the global_model keys during aggregation.
            clean_state_dict = {key.replace('_module.', ''): val.clone().detach()
                                for key, val in local_model.state_dict().items()}
            local_weights.append(clean_state_dict)
            # ---------------

            local_losses.append(avg_loss)

            # Update client state for attention mechanism
            client_states[k, 0] = np.log(len(train_loader.dataset))
            client_states[k, 1] = avg_loss
            # Simplified delta loss and drift for simulation
            client_states[k, 2] = avg_loss - client_states[k, 1].item()
            client_states[k, 3] = torch.rand(1).item() # Mock drift

        # 4. Attention-Gated Global Aggregation (L3)
        with torch.no_grad():
            attention_weights = attention_module(client_states, ATTENTION_TEMP)

            global_state = global_model.state_dict()
            for key in global_state.keys():
                global_state[key] = torch.stack(
                    [local_weights[k][key].float() * attention_weights[k] for k in range(NUM_CLIENTS)]
                ).sum(dim=0)

            global_model.load_state_dict(global_state)

        print(f"Round {round_num+1}/{ROUNDS} completed | Max Privacy Epsilon Expended: ~{round_num * 0.06:.2f}")

if __name__ == "__main__":
    federated_training_loop()

Initializing RECONFIGMED-CHAIN Federation...


/tmp/ipykernel_15274/640118357.py:121: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Round 1/80 completed | Max Privacy Epsilon Expended: ~0.00
Round 2/80 completed | Max Privacy Epsilon Expended: ~0.06
Round 3/80 completed | Max Privacy Epsilon Expended: ~0.12
Round 4/80 completed | Max Privacy Epsilon Expended: ~0.18
Round 5/80 completed | Max Privacy Epsilon Expended: ~0.24
